In [15]:
from openff.toolkit import Molecule, Topology, ForceField
from openff.interchange import Interchange
from openff.units import unit, Quantity
from openff.toolkit.utils.toolkits import NAGLToolkitWrapper
import numpy as np
import time
import mdtraj as md
import openmm
from openff.toolkit.typing.engines.smirnoff import ForceField
from openff.toolkit.utils import get_data_file_path
from pandas import read_csv
from openff.units.openmm import to_openmm

In [16]:
from openff.interchange.components._packmol import pack_box


In [17]:
mol_smiles = "CS(=O)C" #DMSO
monomer =Molecule.from_smiles(mol_smiles)
monomer.generate_conformers(n_conformers=1)

In [18]:

cubic_box =unit.Quantity (40 * np.eye(3), unit.angstrom)

n_monomer = 500

from openff.interchange.components._packmol import pack_box
packed_topology = pack_box(
    molecules=[monomer],
    number_of_copies=[n_monomer],
    solute=None,
    tolerance=2.0*unit.angstrom,
    box_vectors=cubic_box,
)

In [19]:
from openmm import MonteCarloBarostat
from openmm.unit import atmosphere, kelvin

packed_interchange = Interchange.from_smirnoff(
            force_field=ForceField("openff-2.2.0.offxml", "tip3p.offxml"),
            topology=packed_topology, box=cubic_box)            
system = packed_interchange.to_openmm()

barostat = MonteCarloBarostat(
    1.0 * atmosphere,
    300.0 * kelvin,
    25,
)

system.addForce(barostat)
packed_interchange.minimize()
packed_interchange.to_pdb("pure_dmso.pdb")


/projects/ladu6977/polyzymd/.pixi/envs/cuda-12-4/lib/python3.11/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


In [20]:
# Length of the simulation.
num_steps = 5000000  # number of integration steps to run

# Logging options.
trj_freq = 16667  # number of steps per written trajectory frame
data_freq = 5000  # number of steps per written simulation statistics

# Integration options
time_step = 2 * openmm.unit.femtoseconds  # simulation timestep
temperature = 300 * openmm.unit.kelvin  # simulation temperature
friction = 1 / openmm.unit.picosecond  # friction constant

integrator = openmm.LangevinMiddleIntegrator(temperature, friction, time_step)

In [21]:

# Create Simulation
simulation = openmm.app.Simulation(
    packed_interchange.to_openmm_topology(),
    system,
    integrator,
)

# Set positions from the Interchange object
simulation.context.setPositions(
    to_openmm(packed_interchange.positions)
)
simulation.context.setVelocitiesToTemperature(temperature)

dcd_reporter =openmm.app.DCDReporter("pure_dmso_trajectory.dcd", trj_freq)
state_data_reporter = openmm.app.StateDataReporter(
    "pure_dmso_data.csv",
    data_freq,
    step=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    temperature=True,
    volume=True,
    density=True,
    speed=True,
    time=True,
    elapsedTime=True,
    
)
simulation.reporters.append(dcd_reporter)
simulation.reporters.append(state_data_reporter)

In [22]:
print("Starting simulation")
start = time.process_time()

# Run the simulation
simulation.step(num_steps)

end = time.process_time()
print(f"Elapsed time {end - start} seconds")
print("Done!")

Starting simulation
Elapsed time 1374.46599803 seconds
Done!
